# ML Exercise 01 — Exploratory Data Analysis
## Hotel Booking Demand Dataset

**Course:** Data Science & Machine Learning  
**Dataset:** Hotel Booking Demand (`jessemostipak/hotel-booking-demand` on Kaggle)  
**Goal:** Understand the dataset structure, quality, and target distribution before any modelling.

---

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import os
import warnings

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Plot aesthetics ───────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 4)})
warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")
print(f"  pandas  : {pd.__version__}")
print(f"  numpy   : {np.__version__}")
print(f"  seaborn : {sns.__version__}")


---
## 1. Dataset Loading and Initial Inspection (Task 1)

In [ ]:
# ── Dataset loader ────────────────────────────────────────────────────────────
# Tries Kaggle path first, then local ./data/ path.
# Works unchanged in both environments.

KAGGLE_PATH = "/kaggle/input/hotel-booking-demand/hotel_bookings.csv"
LOCAL_PATH  = "./data/hotel_bookings.csv"

if os.path.exists(KAGGLE_PATH):
    DATA_PATH = KAGGLE_PATH
    print(f"Running on Kaggle — loading from:\n  {DATA_PATH}")
elif os.path.exists(LOCAL_PATH):
    DATA_PATH = LOCAL_PATH
    print(f"Running locally — loading from:\n  {DATA_PATH}")
else:
    raise FileNotFoundError(
        "\n[ERROR] Dataset not found.\n"
        "  On Kaggle : Add 'jessemostipak/hotel-booking-demand' as an input dataset.\n"
        "  Locally   : Place 'hotel_bookings.csv' inside the ./data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print(f"\nDataset loaded successfully!")


In [ ]:
# ── 1.1  Shape ────────────────────────────────────────────────────────────────
n_rows, n_cols = df.shape
print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(f"  Rows    : {n_rows:,}")
print(f"  Columns : {n_cols}")


In [ ]:
# ── 1.2  Column names and dtypes ──────────────────────────────────────────────
print("=" * 60)
print("COLUMN NAMES AND DATA TYPES")
print("=" * 60)
dtype_df = pd.DataFrame({
    "Column" : df.columns,
    "Dtype"  : df.dtypes.values
})
print(dtype_df.to_string(index=False))


In [ ]:
# ── 1.3  df.info() ────────────────────────────────────────────────────────────
print("=" * 60)
print("df.info() OUTPUT")
print("=" * 60)
df.info()


In [ ]:
# ── 1.4  Memory usage ─────────────────────────────────────────────────────────
mem_bytes = df.memory_usage(deep=True).sum()
mem_mb    = mem_bytes / (1024 ** 2)
print(f"Memory usage : {mem_bytes:,} bytes  ({mem_mb:.2f} MB)")


In [ ]:
# ── 1.5  Numerical vs Categorical columns ─────────────────────────────────────
# Using pandas dtype inference:
#   numerical   → int64, float64, etc.
#   categorical → object (string), category

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("=" * 60)
print(f"NUMERICAL columns  ({len(num_cols)}):")
print("=" * 60)
for c in num_cols:
    print(f"  {c}")

print()
print("=" * 60)
print(f"CATEGORICAL columns ({len(cat_cols)}):")
print("=" * 60)
for c in cat_cols:
    print(f"  {c}")


In [ ]:
# ── 1.6  Target variable ──────────────────────────────────────────────────────
target_col = "is_canceled"
print("=" * 60)
print("TARGET VARIABLE")
print("=" * 60)
print(f"  Column : '{target_col}'")
print(f"  Dtype  : {df[target_col].dtype}")
print(f"  Classes (value counts):")
print(df[target_col].value_counts().to_string())


In [ ]:
# ── 1.7  Duplicate rows ───────────────────────────────────────────────────────
n_duplicates = df.duplicated().sum()
print(f"Number of exact duplicate rows : {n_duplicates:,}")


In [ ]:
# ── 1.8  Dataset Overview Table ───────────────────────────────────────────────
# One clean summary table — good for the assignment report.

total_missing    = df.isnull().sum().sum()
n_target_classes = df[target_col].nunique()
n_input_features = n_cols - 1  # all columns except target

# Numerical/categorical counts for input features only
n_num_input = len([c for c in num_cols if c != target_col])
n_cat_input = len(cat_cols)   # target is numeric so cat_cols has no target

overview = pd.DataFrame({
    "Property": [
        "Number of Observations",
        "Number of Input Features",
        "Number of Numerical Features",
        "Number of Categorical Features",
        "Number of Missing Values (total)",
        "Number of Duplicate Observations",
        "Dataset Memory Usage",
        "Target Variable",
        "Number of Target Classes",
    ],
    "Result": [
        f"{n_rows:,}",
        f"{n_input_features}",
        f"{n_num_input}",
        f"{n_cat_input}",
        f"{total_missing:,}",
        f"{n_duplicates:,}",
        f"{mem_mb:.2f} MB",
        target_col,
        f"{n_target_classes}",
    ]
})

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
display(overview.style.set_caption("Dataset Overview Table").hide(axis="index"))


---
## 2. Data Dictionary (Task 2)

In [ ]:
# ── 2.1  Human-written descriptions for all 32 columns ───────────────────────
# Source: Kaggle dataset documentation + hotel industry conventions.
# Leakage-prone columns are flagged explicitly with reasons.
# company and agent flagged as high-missing, high-cardinality Identifier-like.

col_meta = {
    # Feature                          : (Description, Feature Category)
    "hotel"                            : ("Type of hotel: 'Resort Hotel' or 'City Hotel'.",
                                          "Categorical"),
    "is_canceled"                      : ("Target variable: 1 if booking was cancelled, 0 otherwise.",
                                          "Binary"),
    "lead_time"                        : ("Number of days between booking date and arrival date.",
                                          "Numerical"),
    "arrival_date_year"                : ("Year of the arrival date (2015, 2016, or 2017).",
                                          "Ordinal"),
    "arrival_date_month"               : ("Month of arrival as a string (e.g., 'July').",
                                          "Temporal"),
    "arrival_date_week_number"         : ("ISO week number of the year for the arrival date.",
                                          "Temporal"),
    "arrival_date_day_of_month"        : ("Day of the month of the arrival date (1–31).",
                                          "Temporal"),
    "stays_in_weekend_nights"          : ("Number of Saturday/Sunday nights included in the stay.",
                                          "Numerical"),
    "stays_in_week_nights"             : ("Number of Monday–Friday nights included in the stay.",
                                          "Numerical"),
    "adults"                           : ("Number of adults included in the booking.",
                                          "Numerical"),
    "children"                         : ("Number of children included in the booking (can be NaN).",
                                          "Numerical"),
    "babies"                           : ("Number of babies included in the booking.",
                                          "Numerical"),
    "meal"                             : ("Meal plan booked: BB (Bed & Breakfast), HB, FB, SC/Undefined.",
                                          "Categorical"),
    "country"                          : ("Country of origin of the guest (ISO 3166-1 alpha-3 code).",
                                          "Categorical"),
    "market_segment"                   : ("Market segment designation (e.g., Online TA, Direct, Corporate).",
                                          "Categorical"),
    "distribution_channel"             : ("Booking distribution channel (e.g., TA/TO, Direct, Corporate).",
                                          "Categorical"),
    "is_repeated_guest"                : ("Flag: 1 if the guest previously stayed at this hotel, 0 otherwise.",
                                          "Binary"),
    "previous_cancellations"           : ("Number of prior bookings by this guest that were cancelled.",
                                          "Numerical"),
    "previous_bookings_not_canceled"   : ("Number of prior bookings by this guest that were NOT cancelled.",
                                          "Numerical"),
    "reserved_room_type"               : ("Code of the room type reserved (anonymised letters A–P).",
                                          "Categorical"),
    "assigned_room_type"               : ("Code of the room type actually assigned at check-in.",
                                          "Categorical"),
    "booking_changes"                  : ("Number of amendments made to the booking before check-in.",
                                          "Numerical"),
    "deposit_type"                     : ("Deposit policy: No Deposit, Non-Refund, or Refundable.",
                                          "Categorical"),
    "agent"                            : ("ID of the travel agency that made the booking — high missing, high cardinality.",
                                          "Identifier-like"),
    "company"                          : ("ID of the company that made the booking — very high missing, high cardinality.",
                                          "Identifier-like"),
    "days_in_waiting_list"             : ("Number of days the booking spent on the waiting list before confirmation.",
                                          "Numerical"),
    "customer_type"                    : ("Type of booking: Transient, Contract, Transient-Party, or Group.",
                                          "Categorical"),
    "adr"                              : ("Average Daily Rate — mean room revenue per occupied room per day.",
                                          "Numerical"),
    "required_car_parking_spaces"      : ("Number of car parking spaces requested by the guest.",
                                          "Numerical"),
    "total_of_special_requests"        : ("Total count of special requests made by the guest.",
                                          "Numerical"),
    "reservation_status"               : (
        "[LEAKAGE] Final reservation status (Check-Out, Canceled, No-Show) — "
        "recorded AFTER the outcome is known, directly encodes the target.",
        "Leakage-prone"
    ),
    "reservation_status_date"          : (
        "[LEAKAGE] Date of the last reservation status update — "
        "set at cancellation/check-out time, post-outcome information.",
        "Leakage-prone"
    ),
}

print(f"Descriptions written for {len(col_meta)} columns.")
print(f"Dataset has {n_cols} columns.")
assert len(col_meta) == n_cols, "Mismatch! Check col_meta dictionary."
print("Column count matches — OK.")


In [ ]:
# ── 2.2  Build the Data Dictionary DataFrame ──────────────────────────────────
# Unique Values, Missing Count, Missing % are computed from actual data — not hardcoded.

rows = []
for col in df.columns:
    description, feature_category = col_meta[col]

    dtype_str     = str(df[col].dtype)
    unique_vals   = df[col].nunique(dropna=False)   # count NaN as distinct
    missing_count = df[col].isnull().sum()
    missing_pct   = (missing_count / len(df)) * 100

    # Proposed action logic
    if feature_category == "Leakage-prone":
        action = "DROP before modelling — post-outcome leakage."
    elif feature_category == "Identifier-like":
        action = "Convert to binary (present/absent) flag or DROP — too many unique IDs."
    elif missing_pct > 50:
        action = "Investigate — consider dropping if missingness is not informative."
    elif missing_pct > 0:
        action = "Impute (median for numeric, mode/constant for categorical)."
    elif feature_category == "Temporal":
        action = "Encode cyclically or as ordinal; consider combining into single date."
    elif feature_category == "Binary":
        action = "Keep as-is; verify 0/1 encoding."
    elif feature_category == "Numerical":
        action = "Check for outliers; scale if needed for linear models."
    elif feature_category == "Categorical":
        action = "Encode (one-hot for low cardinality, target-encode for high cardinality)."
    elif feature_category == "Ordinal":
        action = "Encode as integers preserving order."
    else:
        action = "Review and decide."

    rows.append({
        "Feature"          : col,
        "Description"      : description,
        "Data Type"        : dtype_str,
        "Feature Category" : feature_category,
        "Unique Values"    : unique_vals,
        "Missing Count"    : missing_count,
        "Missing %"        : round(missing_pct, 2),
        "Proposed Action"  : action,
    })

data_dict = pd.DataFrame(rows)

# Sort by Missing % descending — high-missing columns appear first
data_dict = data_dict.sort_values("Missing %", ascending=False).reset_index(drop=True)

print(f"Data Dictionary built: {len(data_dict)} rows x {len(data_dict.columns)} columns")


In [ ]:
# ── 2.3  Display the full Data Dictionary (no truncation) ─────────────────────
# pd.option_context is scoped — does not affect any other cells.

with pd.option_context(
    "display.max_rows",    None,
    "display.max_columns", None,
    "display.max_colwidth", 120,
):
    display(
        data_dict.style
        .set_caption("Data Dictionary — Hotel Booking Demand Dataset")
        .hide(axis="index")
        .set_properties(**{"text-align": "left"})
        .highlight_between(
            subset=["Missing %"],
            left=10, right=100,
            color="#ffe0e0"   # light red for high-missing rows
        )
    )


In [ ]:
# ── 2.4  Quick flag summary ───────────────────────────────────────────────────
print("=" * 60)
print("LEAKAGE-PRONE columns (must drop before modelling):")
print("=" * 60)
leakage_cols = data_dict.loc[
    data_dict["Feature Category"] == "Leakage-prone", "Feature"
].tolist()
for c in leakage_cols:
    print(f"  x  {c}")

print()
print("=" * 60)
print("IDENTIFIER-LIKE columns (high missing / high cardinality):")
print("=" * 60)
id_rows = data_dict[data_dict["Feature Category"] == "Identifier-like"]
for _, row in id_rows.iterrows():
    print(f"  !  {row['Feature']:<35}  Missing: {row['Missing %']:.1f}%   "
          f"Unique IDs: {row['Unique Values']}")


---
## 3. Train-Test Split for Leakage-Free EDA (Task 3)

In [ ]:
# ── 3.1  Separate features (X) and target (y) ─────────────────────────────────
X = df.drop(columns=[target_col])
y = df[target_col]

print(f"X shape : {X.shape}  (all columns except '{target_col}')")
print(f"y shape : {y.shape}  ('{target_col}' column only)")


In [ ]:
# ── 3.2  Stratified train-test split ─────────────────────────────────────────
# stratify=y ensures both splits keep the same class ratio as the full dataset.
# test_size=0.2 -> 80% train, 20% test.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Split sizes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape}")
print(f"  y_test  : {y_test.shape}")


In [ ]:
# ── 3.3  Class distribution — full dataset, train, test ───────────────────────

def class_distribution(series, label):
    """Print count and % of each class for a Series."""
    counts = series.value_counts().sort_index()
    pcts   = series.value_counts(normalize=True).sort_index() * 100
    print(f"  {label} (n={len(series):,}):")
    for cls in counts.index:
        print(f"    Class {cls} -> {counts[cls]:>6,}  ({pcts[cls]:.2f}%)")

print("=" * 60)
print("TARGET CLASS DISTRIBUTION")
print("=" * 60)
class_distribution(y,       "Full dataset")
print()
class_distribution(y_train, "y_train")
print()
class_distribution(y_test,  "y_test")


In [ ]:
# ── 3.4  Stratification quality check (PASS/FAIL) ────────────────────────────
# Verify class proportions in train and test are within 0.5 pp of the full dataset.

TOLERANCE_PP = 0.5   # percentage points

full_pos_pct  = (y == 1).mean() * 100
train_pos_pct = (y_train == 1).mean() * 100
test_pos_pct  = (y_test  == 1).mean() * 100

train_diff = abs(train_pos_pct - full_pos_pct)
test_diff  = abs(test_pos_pct  - full_pos_pct)

train_ok = train_diff <= TOLERANCE_PP
test_ok  = test_diff  <= TOLERANCE_PP

print("=" * 60)
print(f"Stratification check (tolerance = +/-{TOLERANCE_PP} pp)")
print("=" * 60)
print(f"  Full dataset cancellation rate : {full_pos_pct:.3f}%")
print(f"  y_train cancellation rate      : {train_pos_pct:.3f}%  (diff = {train_diff:.3f} pp)  "
      f"{'PASS' if train_ok else 'FAIL'}")
print(f"  y_test  cancellation rate      : {test_pos_pct:.3f}%  (diff = {test_diff:.3f} pp)  "
      f"{'PASS' if test_ok else 'FAIL'}")
print()

if train_ok and test_ok:
    print("[OVERALL] PASS — Stratification is working correctly.")
else:
    print("[OVERALL] FAIL — Class proportions deviate more than allowed. Check split parameters.")
    raise AssertionError("Stratification check failed.")


In [ ]:
# ── 3.5  Recombine into train_df ──────────────────────────────────────────────
# train_df = X_train + y_train merged back into a single DataFrame.
# All subsequent EDA phases (Tasks 4-9) will operate on train_df only,
# keeping the test set completely unseen to prevent information leakage.

train_df = X_train.copy()
train_df[target_col] = y_train.values   # re-attach target column

# Reset index so rows are numbered 0, 1, 2, ...
train_df = train_df.reset_index(drop=True)

print("train_df assembled successfully.")
print(f"  Shape   : {train_df.shape}")
print(f"  Columns : {list(train_df.columns[:5])} ... '{target_col}' (appended as last column)")
print()
print("NOTE: All future EDA (Tasks 4-9) will use 'train_df', not 'df'.")
print("      'X_test' and 'y_test' are locked away — do not analyse them.")


In [ ]:
# ── 3.6  Quick sanity check on train_df ──────────────────────────────────────
print("train_df head (first 3 rows):")
display(train_df.head(3))

print()
print("train_df target distribution:")
class_distribution(train_df[target_col], "train_df")


---
## Notebook continues in Phase 2 — Data Quality Analysis (Tasks 4-9)